# Chapter 1 — Introduction to Data Analytics
**MADT6004 · Brew Lab BKK case**

The COO of Brew Lab BKK, **Khun Ploy**, has questions. To answer them, you need to know which kind of analytics each question calls for.

Four types of analytics:

| Type | Question it answers | Example for Brew Lab |
|---|---|---|
| **Descriptive** | What happened? | How much revenue did each branch generate? |
| **Diagnostic** | Why did it happen? | Which factors are associated with high revenue? |
| **Predictive** | What will happen? | How much will SKU X sell next month? |
| **Prescriptive** | What should we do? | Which lapsed members should get a comeback voucher? |

In this notebook you'll **practice descriptive analytics** — getting the lay of the land before any modeling.


## 0. Bootstrap (Colab + local)

In [ ]:
# Bootstrap — make sure brewlab.db is available, both locally and in Colab.
import os
DB_CANDIDATES = [
    "../../Integrated Data Analytics Exercise/data/brewlab.db",
    "MADT6004/Integrated Data Analytics Exercise/data/brewlab.db",
]
DB_PATH = next((p for p in DB_CANDIDATES if os.path.exists(p)), None)
if DB_PATH is None:
    if not os.path.exists("MADT6004"):
        os.system("git clone -q https://github.com/thanachart/MADT6004.git")
    os.system("pip install -q -r 'MADT6004/Integrated Data Analytics Exercise/requirements.txt'")
    DB_PATH = "MADT6004/Integrated Data Analytics Exercise/data/brewlab.db"
print("DB:", DB_PATH)


## 1. Setup
Load packages and connect to the database.

In [ ]:
import sqlite3
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings("ignore")
sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 100

conn = sqlite3.connect(DB_PATH)
print("Tables:", [r[0] for r in conn.execute("SELECT name FROM sqlite_master WHERE type='table'").fetchall()])


## 2. How big is each table?
Run a quick row-count of every table to understand the scale of the data.

In [ ]:
tables = [r[0] for r in conn.execute("SELECT name FROM sqlite_master WHERE type='table'")]
for t in tables:
    n = conn.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0]
    print(f"  {t:25s} {n:>10,}")


## 3. Top branches by total revenue
Aggregate `transactions` to branch level and sort.

In [ ]:
q = """
SELECT b.name AS branch, b.district,
       COUNT(*) AS orders,
       ROUND(SUM(t.total), 0) AS revenue
FROM transactions t JOIN branches b ON t.branch_id = b.branch_id
GROUP BY b.branch_id
ORDER BY revenue DESC
"""
top = pd.read_sql(q, conn)
print(top)


## 4. Revenue mix by channel
Brew Lab serves customers through four channels — dine-in, takeaway, app pickup, delivery. How does each channel contribute to revenue?

In [ ]:
q = """
SELECT channel,
       COUNT(*) AS orders,
       ROUND(SUM(total), 0) AS revenue,
       ROUND(AVG(total), 1) AS avg_ticket
FROM transactions
GROUP BY channel
ORDER BY revenue DESC
"""
mix = pd.read_sql(q, conn)
print(mix)

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(mix["channel"], mix["revenue"], color="#0891B2")
ax.set_ylabel("Revenue (THB)")
ax.set_title("Revenue by channel")
plt.tight_layout(); plt.show()


## 5. Map each business question to an analytics type
For each of Khun Ploy's questions below, decide whether it is descriptive, diagnostic, predictive, or prescriptive.

1. *What was last month's average ticket size at Thonglor?*
2. *Why does the Bangna branch underperform on weekends?*
3. *How many lattes will Asoke sell next Saturday?*
4. *Which lapsed members should we send the Comeback 50 voucher to?*

(No code — answer in a text cell.)


## Discussion prompts
1. Of the four analytics types, which one delivers the most business value for a chain like Brew Lab? Why?
2. You ran descriptive queries today. What follow-up questions do they raise that descriptive analytics alone can't answer?
3. Why is descriptive analytics still the foundation for the other three types?
